# 🔬 离线调试：`agent/run` — 走完整生产路径，无需任何 Server

**生产路径完整复现**：

```
RunAgentInput (原始 payload)
      ↓
_inject_state()          ← 解析 headers / context → 注入 org_name, layer, project_slug
      ↓
LangGraphAGUIAgent.run() ← ag-ui message 转换 + MemorySaver + event stream
      ↓
admin_graph.ainvoke      ← LangGraph
```

只 mock 一层：`agents.shared.make_client`（GraphQL → fake data）  
HTTP headers 通过 `MagicMock` 注入，完全无需启动 FastAPI server。

In [ ]:
import sys, os, json, uuid

AGENT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(AGENT_ROOT, '.env'))

from logging_setup import setup_logging
setup_logging()

import config
print('AGENT_ROOT:', AGENT_ROOT)
print('LLM_MODEL :', config.LLM_MODEL)

## 1. Mock GraphQL Client

In [ ]:
from unittest.mock import AsyncMock, MagicMock, patch

FAKE_PROJECTS = [
    {"id": "p1", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE"},
    {"id": "p2", "slug": "abcde",        "title": "abcde",       "description": "", "status": "ACTIVE"},
    {"id": "p3", "slug": "pro1",         "title": "pro1",        "description": "", "status": "ACTIVE"},
]

def make_fake_client(state=None):
    c = MagicMock()
    c.list_projects    = AsyncMock(return_value={"data": {"projects": FAKE_PROJECTS}})
    c.list_databases   = AsyncMock(return_value={"data": {"listDatabases": {"edges": [
        {"node": {"name": "maindb"}}, {"node": {"name": "analyticsdb"}}
    ]}}})
    c.list_models      = AsyncMock(return_value={"data": {"models": {"items": [
        {"id": "m1", "name": "user",  "title": "用户"},
        {"id": "m2", "name": "order", "title": "订单"},
    ], "hasNextPage": False}}})
    c.get_model_fields = AsyncMock(return_value={"data": {"model": {"fields": []}}})
    c.query_model      = AsyncMock(return_value={"data": {"queryModel": {"items": [], "hasNextPage": False}}})
    c.nl2filter        = AsyncMock(return_value={"data": {"nl2filter": {"filter": "{}", "explanation": "mock"}}})
    return c

_patcher = patch('agents.shared.make_client', side_effect=make_fake_client)
_patcher.start()
print('✅ make_client 已 mock')

## 2. 构造 RunAgentInput + 伪造 Request headers

原始 `agent/run` payload 直接映射到 `RunAgentInput`。  
`_inject_state` 从 HTTP headers 读 `X-Org-Name` / `Authorization`，用 `MagicMock` 伪造即可。

In [ ]:
from ag_ui_langgraph.endpoint import RunAgentInput
from ag_ui.core.types import UserMessage, Context, Tool

# ── 原始 agent/run payload（按需修改）─────────────────────────────────
USER_MESSAGE = "列出我所有的项目"
ORG_NAME     = "lukeco"
LAYER        = "org"
PROJECT_SLUG = "Default Project"
# ──────────────────────────────────────────────────────────────────────

input_data = RunAgentInput(
    threadId=str(uuid.uuid4()),
    runId=str(uuid.uuid4()),
    state={},
    messages=[
        UserMessage(role="user", content=USER_MESSAGE),
    ],
    tools=[
        Tool(name="show_toast",         description="向用户显示通知",     parameters={"type": "object", "properties": {"message": {"type": "string"}}, "required": ["message"]}),
        Tool(name="navigate_to_project", description="跳转到指定项目",    parameters={"type": "object", "properties": {"slug":    {"type": "string"}}, "required": ["slug"]}),
        Tool(name="open_create_project", description="打开新建项目表单",  parameters={"type": "object", "properties": {"slug": {"type": "string"}, "title": {"type": "string"}}}),
        Tool(name="highlight_project",   description="高亮指定项目",      parameters={"type": "object", "properties": {"slug": {"type": "string"}, "reason": {"type": "string"}}, "required": ["slug", "reason"]}),
    ],
    context=[
        Context(description="当前 AI 上下文", value=json.dumps({
            "layer":            LAYER,
            "orgName":          ORG_NAME,
            "availableActions": ["navigate_to_project", "open_create_project", "highlight_project", "list_projects", "nl2filter"],
        })),
    ],
    forwardedProps={"projectId": "default", "projectSlug": PROJECT_SLUG, "orgName": ORG_NAME},
)

# ── 伪造 HTTP Request headers（替代真实 FastAPI Request）──────────────
fake_request = MagicMock()
fake_request.headers = {
    "Authorization": "Bearer fake-offline-token",
    "X-Org-Name":    ORG_NAME,        # 生产中由 APISIX 从 JWT 注入
}

print('✅ RunAgentInput 构造完成')
print(f'   threadId : {input_data.threadId}')
print(f'   messages : {[m.content for m in input_data.messages]}')
print(f'   X-Org-Name header: {fake_request.headers["X-Org-Name"]}')

## 3. _inject_state → LangGraphAGUIAgent.run（完整生产路径）

In [ ]:
from copilotkit import LangGraphAGUIAgent
from agents.admin_agent import admin_graph
from main import _inject_state

# ── 1. 构造 agent（与 main.py 完全一致）──────────────────────────────
agent = LangGraphAGUIAgent(
    name="modelcraft_admin_agent",
    description="ModelCraft AI 助手（管理员版）：项目管理、建模、数据查询",
    graph=admin_graph,
)

# ── 2. _inject_state：用 fake_request headers 填充 state ─────────────
injected = _inject_state(input_data, fake_request)

print('_inject_state 结果（state 字段）:')
for k, v in injected.state.items():
    print(f'  {k:15s}: {v}')

# ── 3. agent.run → 收集所有 AG-UI events ─────────────────────────────
events = []
async for event in agent.clone().run(injected):
    events.append(event)

print(f'\n共收到 {len(events)} 个 AG-UI events')

## 4. 查看 AG-UI events

In [ ]:
def safe_dict(e):
    if isinstance(e, dict):
        return e
    return e.model_dump() if hasattr(e, 'model_dump') else vars(e)

for i, e in enumerate(events):
    d = safe_dict(e)
    etype = d.get('type', type(e).__name__)

    if 'TEXT_MESSAGE' in etype or 'text_message' in etype.lower():
        content = d.get('content', d.get('delta', ''))
        if content:
            print(f'[{i:02d}] {etype:35s}  {str(content)[:80]}')
        else:
            print(f'[{i:02d}] {etype}')
    elif 'TOOL' in etype.upper():
        print(f'[{i:02d}] {etype:35s}  {json.dumps(d, ensure_ascii=False)[:100]}')
    else:
        print(f'[{i:02d}] {etype}')

## 5. 提取最终回复文字

In [ ]:
from main import _extract_ai_text

reply_parts = [_extract_ai_text(e) for e in events]
reply = ''.join(p for p in reply_parts if p)

print('─' * 60)
print('🤖 Agent 回复：')
print(reply)
print('─' * 60)

## 6. 换消息重发（不重建 agent，复用同一 thread）

In [ ]:
async def run_message(msg: str, layer: str = "org", project_slug: str = PROJECT_SLUG) -> str:
    new_input = RunAgentInput(
        threadId=str(uuid.uuid4()),
        runId=str(uuid.uuid4()),
        state={},
        messages=[UserMessage(role="user", content=msg)],
        tools=input_data.tools,
        context=[Context(description="当前 AI 上下文", value=json.dumps({
            "layer": layer, "orgName": ORG_NAME,
        }))],
        forwardedProps={"orgName": ORG_NAME, "projectSlug": project_slug},
    )
    injected = _inject_state(new_input, fake_request)
    evts = []
    async for e in agent.clone().run(injected):
        evts.append(e)
    reply = ''.join(p for p in [_extract_ai_text(e) for e in evts] if p)
    print(f'Q: {msg!r}')
    print(f'A: {reply[:120]}\n')
    return reply

await run_message("列出我所有的项目")
await run_message("帮我跳转到 abcde 项目")
await run_message("新建项目，名字叫 test-debug")

## 7. 清理

In [ ]:
_patcher.stop()
print('✅ mock 已恢复')

In [ ]:
_patcher.stop()
# _llm_patcher.stop()  # 如果启用了 LLM mock
print('✅ 所有 mock 已恢复')